Paso 1: código base para construir la pseudo-RGB

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from spectral import open_image

Cargar la HSI

In [ ]:
hdr_path = r"Datos\SB013\SB013\SB013_001\SB013_nrm.hdr"

img = open_image(hdr_path)
cube = np.asarray(img.load(), dtype=np.float32)

wavelengths = np.array([float(w) for w in img.metadata["wavelength"]], dtype=np.float32)

print("cube shape:", cube.shape)
print("wavelengths shape:", wavelengths.shape)
print("primeras longitudes de onda:", wavelengths[:10])

In [ ]:
pseudo_rgb, rgb_info = hsi_to_pseudo_rgb(cube, wavelengths)
plt.figure(figsize=(6,6))
plt.imshow(pseudo_rgb)
plt.axis("off")
plt.show()

In [ ]:
hdr_path = r"Datos\SB013\SB013\SB013_2_001\SB013_2_nrm.hdr"

img = open_image(hdr_path)
cube = np.asarray(img.load(), dtype=np.float32)

wavelengths = np.array([float(w) for w in img.metadata["wavelength"]], dtype=np.float32)

print("cube shape:", cube.shape)
print("wavelengths shape:", wavelengths.shape)
print("primeras longitudes de onda:", wavelengths[:10])

In [ ]:
pseudo_rgb_2, rgb_info = hsi_to_pseudo_rgb(cube, wavelengths)
plt.figure(figsize=(6,6))
plt.imshow(pseudo_rgb_2)
plt.axis("off")
plt.show()

Función para elegir bandas RGB cercanas

In [ ]:
def find_nearest_band(wavelengths, target_nm):
    wavelengths = np.asarray(wavelengths, dtype=float)
    idx = np.argmin(np.abs(wavelengths - target_nm))
    return idx, wavelengths[idx]

Función de normalización robusta

In [ ]:
def robust_normalize(channel, p_low=2, p_high=98):
    ch = channel.astype(np.float32)
    lo = np.percentile(ch, p_low)
    hi = np.percentile(ch, p_high)

    if hi <= lo:
        return np.zeros_like(ch, dtype=np.float32)

    ch = (ch - lo) / (hi - lo)
    ch = np.clip(ch, 0, 1)
    return ch

Construir pseudo-RGB desde la HSI

In [ ]:
def hsi_to_pseudo_rgb(cube, wavelengths, r_nm=650, g_nm=550, b_nm=450):
    r_idx, r_real = find_nearest_band(wavelengths, r_nm)
    g_idx, g_real = find_nearest_band(wavelengths, g_nm)
    b_idx, b_real = find_nearest_band(wavelengths, b_nm)

    r = robust_normalize(cube[:, :, r_idx])
    g = robust_normalize(cube[:, :, g_idx])
    b = robust_normalize(cube[:, :, b_idx])

    rgb = np.stack([r, g, b], axis=-1)

    info = {
        "r_idx": r_idx, "r_nm": r_real,
        "g_idx": g_idx, "g_nm": g_real,
        "b_idx": b_idx, "b_nm": b_real
    }
    return rgb, info

In [ ]:
pseudo_rgb_2, rgb_info = hsi_to_pseudo_rgb(cube, wavelengths)

print(rgb_info)

plt.figure(figsize=(6, 6))
plt.imshow(pseudo_rgb_2)
plt.title(
    f"Pseudo-RGB | "
    f"R={rgb_info['r_nm']:.1f}nm, "
    f"G={rgb_info['g_nm']:.1f}nm, "
    f"B={rgb_info['b_nm']:.1f}nm"
)
plt.axis("off")
plt.show()

Paso 2: cargar la imagen destino

In [ ]:
import cv2
import matplotlib.pyplot as plt

wsi_path = r"Datos\SB013\SB013\WSI\Paraffin.jpg"

wsi_bgr = cv2.imread(wsi_path, cv2.IMREAD_COLOR)
if wsi_bgr is None:
    raise FileNotFoundError(f"No se pudo leer la WSI: {wsi_path}")

wsi_rgb = cv2.cvtColor(wsi_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(6, 6))
plt.imshow(wsi_rgb)
plt.title("WSI / macrofoto (fixed)")
plt.axis("off")
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.imshow(pseudo_rgb)
plt.title("HSI pseudo-RGB (moving)")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(wsi_rgb)
plt.title("WSI (fixed)")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
def resize_max_side(img, max_side=1200):
    h, w = img.shape[:2]
    scale = max_side / max(h, w)
    if scale >= 1:
        return img, 1.0
    new_w = int(w * scale)
    new_h = int(h * scale)
    out = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    return out, scale

wsi_small, wsi_scale = resize_max_side(wsi_rgb, max_side=1200)
hsi_small, hsi_scale = resize_max_side((pseudo_rgb * 255).astype("uint8"), max_side=1200)

plt.figure(figsize=(12,6))
plt.subplot(1,2,1)
plt.imshow(hsi_small)
plt.title("HSI pseudo-RGB reducida")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(wsi_small)
plt.title("WSI reducida")
plt.axis("off")
plt.show()

Técnica deep

In [ ]:
!pip install kornia opencv-python matplotlib

In [ ]:
import cv2
import torch
import kornia as K
import kornia.feature as KF
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
def resize_max_side(img, max_side=1024):
    h, w = img.shape[:2]
    scale = max_side / max(h, w)
    if scale >= 1:
        return img, 1.0
    new_w = int(w * scale)
    new_h = int(h * scale)
    out = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    return out, scale


def rgb_to_gray_tensor(img_rgb):
    # img_rgb: uint8, shape (H, W, 3)
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)   # (H, W)
    gray = gray.astype(np.float32) / 255.0             # [0,1]
    gray = torch.from_numpy(gray)[None, None, :, :]    # (1,1,H,W)
    return gray

In [ ]:
hsi_rgb_u8 = (pseudo_rgb * 255).astype(np.uint8)

hsi_small, _ = resize_max_side(hsi_rgb_u8, max_side=1024)
wsi_small, _ = resize_max_side(wsi_rgb, max_side=1024)

plt.figure(figsize=(12,6))
plt.subplot(1,2,1)
plt.imshow(hsi_small)
plt.title("HSI pseudo-RGB")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(wsi_small)
plt.title("WSI")
plt.axis("off")
plt.show()

In [ ]:
img0 = rgb_to_gray_tensor(hsi_small).to(device)
img1 = rgb_to_gray_tensor(wsi_small).to(device)

print("img0 shape:", img0.shape)
print("img1 shape:", img1.shape)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
matcher = KF.LoFTR(pretrained="outdoor").to(device).eval()

img0 = rgb_to_gray_tensor(hsi_small).to(device)
img1 = rgb_to_gray_tensor(wsi_small).to(device)

with torch.no_grad():
    input_dict = {"image0": img0, "image1": img1}
    correspondences = matcher(input_dict)

mkpts0 = correspondences["keypoints0"].cpu().numpy()
mkpts1 = correspondences["keypoints1"].cpu().numpy()
mconf  = correspondences["confidence"].cpu().numpy()

print("Número de matches:", len(mkpts0))

In [ ]:
def draw_matches(img0, img1, pts0, pts1, max_draw=80):
    n = min(len(pts0), max_draw)
    idx = np.linspace(0, len(pts0)-1, n, dtype=int) if len(pts0) > 0 else []

    p0 = pts0[idx] if len(pts0) > 0 else np.empty((0,2))
    p1 = pts1[idx] if len(pts1) > 0 else np.empty((0,2))

    h0, w0 = img0.shape[:2]
    h1, w1 = img1.shape[:2]
    H = max(h0, h1)
    canvas = np.zeros((H, w0 + w1, 3), dtype=np.uint8)
    canvas[:h0, :w0] = img0
    canvas[:h1, w0:w0+w1] = img1

    for (x0, y0), (x1, y1) in zip(p0, p1):
        c = tuple(np.random.randint(0, 255, 3).tolist())
        pt0 = (int(x0), int(y0))
        pt1 = (int(x1) + w0, int(y1))
        cv2.circle(canvas, pt0, 3, c, -1)
        cv2.circle(canvas, pt1, 3, c, -1)
        cv2.line(canvas, pt0, pt1, c, 1)

    return canvas

vis = draw_matches(hsi_small, wsi_small, mkpts0, mkpts1, max_draw=60)

plt.figure(figsize=(16,8))
plt.imshow(vis)
plt.title("Matches LoFTR")
plt.axis("off")
plt.show()

In [ ]:
if len(mkpts0) >= 4:
    H, inliers = cv2.findHomography(mkpts0, mkpts1, cv2.RANSAC, 5.0)
    print("Homografía estimada:", H is not None)
    if inliers is not None:
        print("Inliers:", int(inliers.sum()), "/", len(inliers))
else:
    print("No hay suficientes matches para homografía.")

In [ ]:
if len(mkpts0) >= 3:
    M_affine, inliers_affine = cv2.estimateAffinePartial2D(
        mkpts0, mkpts1, method=cv2.RANSAC, ransacReprojThreshold=5.0
    )
    print("Affine estimada:", M_affine is not None)
    if inliers_affine is not None:
        print("Inliers affine:", int(inliers_affine.sum()), "/", len(inliers_affine))

In [ ]:
if M_affine is not None:
    h, w = wsi_small.shape[:2]
    warped = cv2.warpAffine(hsi_small, M_affine, (w, h))

    overlay = cv2.addWeighted(wsi_small, 0.5, warped, 0.5, 0)

    plt.figure(figsize=(12,6))
    plt.subplot(1,2,1)
    plt.imshow(warped)
    plt.title("HSI warped")
    plt.axis("off")

    plt.subplot(1,2,2)
    plt.imshow(overlay)
    plt.title("Overlay")
    plt.axis("off")
    plt.show()